In [6]:
import json

# Definimos una lista de productos con características variadas (Variedad)
tienda_ropa = [
    {
        "id": "PROD-001",
        "nombre": "Polera Oversize",
        "precio": 15990,
        "caracteristicas": {"color": "Negro", "material": "Algodón 100%", "tallas": ["S", "M", "L"]},
        "imagen_url": "https://virtualsur.cl/ropa/polera_negra.jpg",
        "stock": 50
    },
    {
        "id": "PROD-002",
        "nombre": "Jeans Slim Fit",
        "precio": 29990,
        "caracteristicas": {"color": "Azul Denim", "calce": "Slim"},
        "imagen_url": "https://virtualsur.cl/ropa/jeans_azul.jpg",
        "stock": 30,
        "temporada": "Invierno 2026"
    },
    {
        "id": "PROD-003",
        "nombre": "Chaqueta Eco-Friendly",
        "precio": 45990,
        "caracteristicas": {"color": "Verde Oliva", "material": "Poliéster Reciclado", "tallas": ["M", "L", "XL"]},
        "imagen_url": "https://virtualsur.cl/ropa/chaqueta_eco.jpg",
        "stock": 20,
        "es_sustentable": True,
        "descuento_estudiante": 15
    }
]

# Guardar como archivo JSON para "importar"
with open('productos_tienda.json', 'w') as f:
    json.dump(tienda_ropa, f, indent=4)

print("Archivo productos_tienda.json creado con 3 productos (incluye Chaqueta Eco-Friendly con campos exclusivos).")

Archivo productos_tienda.json creado con 3 productos (incluye Chaqueta Eco-Friendly con campos exclusivos).


In [7]:
# === INTEGRACION: Importar productos desde el JSON y verificar actualización automática ===

with open('productos_tienda.json', 'r') as f:
    productos_importados = json.load(f)

print(f"Productos importados desde JSON: {len(productos_importados)}")
for p in productos_importados:
    campos = list(p.keys())
    print(f"  - {p['nombre']} | Campos: {campos}")

Productos importados desde JSON: 3
  - Polera Oversize | Campos: ['id', 'nombre', 'precio', 'caracteristicas', 'imagen_url', 'stock']
  - Jeans Slim Fit | Campos: ['id', 'nombre', 'precio', 'caracteristicas', 'imagen_url', 'stock', 'temporada']
  - Chaqueta Eco-Friendly | Campos: ['id', 'nombre', 'precio', 'caracteristicas', 'imagen_url', 'stock', 'es_sustentable', 'descuento_estudiante']


In [8]:
# Cargamos la "base de datos" NoSQL simulada con los productos importados
db_mongo = {
    "clientes":[],
    "productos": productos_importados,
    "ventas": []
}

print(f" Coleccion 'productos' cargada con {len(db_mongo['productos'])} articulos.")

 Coleccion 'productos' cargada con 3 articulos.


In [9]:
# Simulando una venta de un cliente
nueva_venta = {
    "n_pedido": "VENTA-5501",
    "cliente": {"nombre": "Francisco Arias", "email": "f.arias@uach.cl"},
    "items": [
        {"prod_id": "PROD-001", "cantidad": 2, "subtotal": 31980}
    ],
    "total": 31980,
    "fecha": "2026-04-15"
}

db_mongo["ventas"].append(nueva_venta)
print("Venta registrada con éxito en la colección 'ventas'.")

Venta registrada con éxito en la colección 'ventas'.


In [10]:
from IPython.display import HTML

# Generamos el HTML dinámicamente desde nuestra base de datos NoSQL
html_productos = "<div style='display: flex; gap: 20px; flex-wrap: wrap;'>"

for p in db_mongo["productos"]:
    # Badges opcionales: solo aparecen si el producto tiene esos campos
    badges = ""
    if p.get("es_sustentable"):
        badges += "<span style='background:#28a745;color:white;padding:2px 8px;border-radius:10px;font-size:11px;'>Sustentable</span> "
    if p.get("descuento_estudiante"):
        badges += f"<span style='background:#ffc107;color:black;padding:2px 8px;border-radius:10px;font-size:11px;'>-{p['descuento_estudiante']}% Estudiante</span>"

    html_productos += f"""
    <div style='border: 1px solid #ddd; padding: 10px; border-radius: 10px; width: 200px; text-align: center;'>
        <img src='{p['imagen_url']}' style='width: 100%; border-radius: 5px;'>
        <h4>{p['nombre']}</h4>
        <p style='color: green; font-weight: bold;'>${p['precio']}</p>
        <div>{badges}</div>
        <button style='background: #007bff; color: white; border: none; padding: 5px 10px; border-radius: 5px; margin-top:5px;'>Ver detalles</button>
    </div>
    """

html_productos += "</div>"

display(HTML(html_productos))

---
## Análisis: Agregar Talla de Calzado para Zapatos

**Pregunta:** *"Si mañana queremos vender zapatos y necesitamos agregar la 'Talla de Calzado', ¿tuvimos que modificar la estructura de los productos anteriores?"*

**Respuesta: No.**

En un modelo NoSQL orientado a documentos (como MongoDB o nuestro JSON simulado), cada documento es **independiente** y puede tener un esquema distinto. Esto se demostró en este mismo ejercicio:

| Producto | Campos exclusivos |
|---|---|
| Polera Oversize | `caracteristicas.tallas`, `caracteristicas.material` |
| Jeans Slim Fit | `caracteristicas.calce`, `temporada` |
| Chaqueta Eco-Friendly | `es_sustentable`, `descuento_estudiante` |

Para agregar zapatos con `talla_calzado`, simplemente se inserta un nuevo documento:

```json
{
    "id": "PROD-004",
    "nombre": "Zapatilla Running",
    "precio": 59990,
    "caracteristicas": {"color": "Blanco"},
    "talla_calzado": 42,
    "stock": 15
}
```

**Los productos anteriores no se tocan.** No hay que agregar `talla_calzado: null` a la polera ni a los jeans. La Página de Ventas sigue funcionando porque el ciclo `for` itera sobre cualquier cantidad de productos y los campos opcionales se manejan con `.get()` (si existe, se muestra; si no, se ignora).

Esto es la esencia de la **Variedad** en Big Data: datos con estructuras heterogéneas coexisten sin forzar un esquema rígido. En SQL relacional, agregar `talla_calzado` requeriría un `ALTER TABLE` que afecta a **todas** las filas existentes.